# Дообучение модели Visual Prolog в Google Colab

Ноутбук дообучает модель **Qwen2.5-Coder-Instruct** писать программы на **Visual Prolog 11** и решать алгоритмические задачи.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **GPU T4**, затем «Среда выполнения» → **«Выполнить все»**. Дальше ноутбук всё сделает сам: установит пакеты, скачает данные, обучит модель, проверит её на задачах и сохранит результат на Google Диск.

| Шаг | Что происходит |
|---|---|
| 1. Проверка | есть ли видеокарта |
| 2. Установка | Unsloth, TRL, datasets |
| 3. Параметры | модель, число эпох, скорость обучения — можно поменять в форме |
| 4. Данные | открытая часть датасета из репозитория GitHub и (если есть) полный архив с Google Диска |
| 5. Подготовка | формат «вопрос → ответ», отложенные задачи для честной проверки |
| 6. Обучение | QLoRA, график потерь |
| 7. Проверка | ответы на бенчмарк (25 задач, которых нет в обучении) и на отложенные алгоритмы — до и после обучения |
| 8. Сохранение | адаптер и ответы на Google Диск, по желанию — модель целиком и GGUF для Ollama |

**Что такое QLoRA.** Сама модель хранится в 4-битном виде и не меняется, а рядом обучаются небольшие матрицы-«поправки» (LoRA-адаптер, около 1 % параметров). Так модель на 7 млрд параметров помещается в бесплатную видеокарту T4 (16 ГБ). Полное дообучение всех весов на T4 не помещается даже для модели 1,5B.

**Чему учим.** Синтаксис Visual Prolog модель уже знает неплохо, поэтому главное в данных — **алгоритмы**: задача, идея решения простыми словами и законченная программа, проверенная компилятором (`data/dataset/algorithms.jsonl`). Остальные данные (учебники, wiki, лабораторные, PFC) закрепляют язык и библиотеку.

## 1. Видеокарта

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Нет GPU: «Среда выполнения» → «Сменить среду выполнения» → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Установка пакетов

Занимает 2–3 минуты. Предупреждения о версиях пакетов можно не читать.

In [ ]:
%%capture
!pip install -q unsloth

## 3. Параметры

Значения по умолчанию подобраны для бесплатной T4. Если Colab отключается раньше конца обучения, выберите модель поменьше (`3B` или `1.5B`) или уменьшите число эпох.

- **MODEL** — базовая модель. 7B пишет код заметно лучше, но учится примерно в 4 раза дольше 1.5B.
- **EPOCHS** — сколько раз модель увидит весь датасет. Алгоритмы дополнительно повторяются `ALGO_REPEAT` раз за эпоху: их мало, а они важнее всего.
- **USE_PFC** — брать ли часть из библиотеки PFC. Её нет в открытом репозитории (лицензия запрещает распространять исходники); нужен архив `vp_dataset_full.zip` на вашем Google Диске.
- **USE_LEGACY** — брать ли код старого Visual Prolog 5 / Turbo Prolog. По умолчанию нет: он отличается от VP 11 и путает модель.
- **DATA_MODE** — какие данные брать. `all` — всё (для лучшей модели). `textbook` и `github` — для проверки гипотезы исследования: учебные примеры «пояснение → код» против сырого кода с GitHub (обучите по разу в каждом режиме с одинаковыми остальными параметрами и сравните ответы).
- **EVAL_BASE** — до обучения дать те же задачи исходной модели. Так видно, что именно дало обучение; добавляет 10–20 минут.

In [ ]:
#@title Параметры обучения { display-mode: "form" }
MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"  #@param ["unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit", "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit", "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit"]
EPOCHS = 2  #@param {type:"integer"}
LEARNING_RATE = 2e-4  #@param {type:"number"}
LORA_RANK = 32  #@param [16, 32, 64] {type:"raw"}
MAX_SEQ_LENGTH = 2048  #@param [1024, 2048, 4096] {type:"raw"}
ALGO_REPEAT = 3  #@param {type:"integer"}
USE_PFC = True  #@param {type:"boolean"}
USE_LEGACY = False  #@param {type:"boolean"}
DATA_MODE = "all"  #@param ["all", "textbook", "github"]
EVAL_BASE = True  #@param {type:"boolean"}
DRIVE_ZIP = "/content/drive/MyDrive/vp_dataset_full.zip"  #@param {type:"string"}
OUT_DIR = "/content/drive/MyDrive/visual-prolog-llm"  #@param {type:"string"}
RUN_NAME = "v3_algorithms"  #@param {type:"string"}
SAVE_MERGED = False  #@param {type:"boolean"}
SAVE_GGUF = False  #@param {type:"boolean"}
SEED = 3407
print("Модель:", MODEL, "| эпох:", EPOCHS, "| r =", LORA_RANK)

## 4. Данные

Открытая часть берётся из репозитория GitHub. Google Диск подключается, чтобы взять полный архив (с частью PFC) и сохранить туда результат — Colab попросит разрешение.

In [ ]:
import os, json, zipfile, pathlib
from google.colab import drive
drive.mount("/content/drive")

!rm -rf /content/visual-prolog-llm && git clone -q --depth 1 https://github.com/DarkSwordman999/visual-prolog-llm /content/visual-prolog-llm
REPO = pathlib.Path("/content/visual-prolog-llm")
DATA = REPO / "data" / "dataset"

if USE_PFC:
    if os.path.exists(DRIVE_ZIP):
        with zipfile.ZipFile(DRIVE_ZIP) as z:
            for name in z.namelist():
                if name.endswith(".jsonl") and "/pfc_" in name:
                    (DATA / pathlib.Path(name).name).write_bytes(z.read(name))
        print("PFC взята из архива на Диске")
    else:
        print("Архива нет:", DRIVE_ZIP, "— учимся без PFC")
        USE_PFC = False

def load(name):
    path = DATA / f"{name}.jsonl"
    return [json.loads(l) for l in open(path, encoding="utf-8")] if path.exists() else []

parts = {name: load(name) for name in ["algorithms", "textbook_pairs", "textbook_code", "github_code", "pfc_pairs", "pfc_code"]}
for name, rows in parts.items():
    print(f"{name:15} {len(rows):5} записей")

## 5. Подготовка примеров

Каждый пример превращается в диалог «пользователь → ассистент» в формате чата модели. Модель учится только на **ответе** ассистента, а не на вопросе.

- В ответе на задачу-алгоритм сначала идёт **идея решения**, потом программа в блоке ```` ```visualprolog ````. Так модель привыкает сначала думать, потом писать код.
- Часть алгоритмов (каждый 8-й) **откладывается** и в обучение не попадает: по ним после обучения видно, научилась ли модель решать новые задачи, а не пересказывать выученные.

In [ ]:
import random
random.seed(SEED)

SYSTEM = ("Ты — опытный программист на Visual Prolog 11. Отвечай по-русски. Сначала кратко объясни идею решения, "
          "затем приведи законченную программу в блоке ```visualprolog. Консольная программа — это файл main.pro: "
          "implement main, раздел class predicates для своих предикатов, предикат run() и goal console::runUtf8(main::run).")

def answer(row):
    code_block = "```visualprolog\n" + row["output"].strip() + "\n```"
    if row.get("source") == "algorithms":
        return "Идея: " + row["explanation"].strip() + "\n\n" + code_block
    return code_block

def keep(row):
    return USE_LEGACY or row.get("dialect") != "legacy"

algos = parts["algorithms"]
held_out = [r for i, r in enumerate(algos) if i % 8 == 7]
train_algos = [r for i, r in enumerate(algos) if i % 8 != 7]

SOURCES = {
    "all": ["textbook_pairs", "textbook_code", "github_code"] + (["pfc_pairs", "pfc_code"] if USE_PFC else []),
    "textbook": ["textbook_pairs", "textbook_code"],
    "github": ["github_code"],
}
rows = train_algos * ALGO_REPEAT if DATA_MODE == "all" else []
for name in SOURCES[DATA_MODE]:
    rows += [r for r in parts[name] if keep(r)]
random.shuffle(rows)

examples = [{"messages": [{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": r["instruction"]},
                          {"role": "assistant", "content": answer(r)}]} for r in rows]
print("Режим данных:", DATA_MODE, "| примеров для обучения:", len(examples), "| отложено алгоритмов:", len(held_out))
print("\nПример:\n", examples[0]["messages"][1]["content"][:300], "\n---\n", examples[0]["messages"][2]["content"][:600])

## 6. Модель и обучение

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()

### Ответы исходной модели

Пока адаптер не обучен, его поправки равны нулю, и модель отвечает как исходная. Её ответы на те же задачи сохраняются для сравнения (если `EVAL_BASE`).

In [ ]:
import time

def ask(question, max_new_tokens=1024):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

bench = json.load(open(REPO / "results" / "benchmark_v2.json", encoding="utf-8"))

def evaluate(label):
    FastLanguageModel.for_inference(model)
    t0, res = time.time(), []
    for t in bench:
        res.append({"model": label, "kind": "benchmark", "id": t["id"], "category": t["category"], "instruction": t["instruction"],
                    "stdin": t.get("stdin", ""), "expected_numbers": t["expected_numbers"], "answer": ask(t["instruction"])})
    for r in held_out:
        res.append({"model": label, "kind": "held_out_algorithm", "id": r["id"], "title": r.get("title", ""), "instruction": r["instruction"],
                    "stdin": r.get("stdin", ""), "expected_output": r.get("expected_output", ""), "answer": ask(r["instruction"])})
    print(f"{label}: {len(res)} ответов за {(time.time() - t0) / 60:.0f} мин")
    return res

answers = evaluate("base") if EVAL_BASE else []
if answers:
    print(answers[0]["instruction"], "\n\n", answers[0]["answer"][:1500])
FastLanguageModel.for_training(model)

In [ ]:
from datasets import Dataset

texts = [tokenizer.apply_chat_template(e["messages"], tokenize=False) for e in examples]
lengths = [len(tokenizer(t).input_ids) for t in texts]
fits = [t for t, n in zip(texts, lengths) if n <= MAX_SEQ_LENGTH]
print(f"Длиннее {MAX_SEQ_LENGTH} токенов и пропущено: {len(texts) - len(fits)}")
print(f"Токенов за эпоху: {sum(n for n in lengths if n <= MAX_SEQ_LENGTH):,}".replace(",", " "))

ds = Dataset.from_dict({"text": fits}).train_test_split(test_size=0.03, seed=SEED)
train_ds, eval_ds = ds["train"], ds["test"]
print("Обучение:", len(train_ds), "| проверка потерь:", len(eval_ds))

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

# в разных версиях TRL параметры называются по-разному
cfg_params = inspect.signature(SFTConfig.__init__).parameters
length_arg = {"max_length" if "max_length" in cfg_params else "max_seq_length": MAX_SEQ_LENGTH}
tok_arg = {"processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters else "tokenizer": tokenizer}

trainer = SFTTrainer(
    model=model, train_dataset=train_ds, eval_dataset=eval_ds, **tok_arg,
    args=SFTConfig(
        output_dir="outputs", dataset_text_field="text", packing=False, **length_arg,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_ratio=0.03,
        optim="adamw_8bit", weight_decay=0.01, seed=SEED,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, eval_strategy="steps", eval_steps=100, save_strategy="steps", save_steps=100,
        save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss",
        report_to="none"))
# учимся только на ответах ассистента
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(f"Готово за {stats.metrics['train_runtime'] / 60:.0f} мин")

In [ ]:
import matplotlib.pyplot as plt
log = trainer.state.log_history
tr = [(x["step"], x["loss"]) for x in log if "loss" in x]
ev = [(x["step"], x["eval_loss"]) for x in log if "eval_loss" in x]
plt.figure(figsize=(8, 4))
plt.plot(*zip(*tr), label="обучение")
if ev:
    plt.plot(*zip(*ev), "o-", label="проверка")
plt.xlabel("шаг"); plt.ylabel("потери"); plt.title("Потери при обучении"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
if ev:
    print("Лучшие потери на проверке:", round(min(v for _, v in ev), 3))

## 7. Проверка

Обученная модель отвечает на 25 задач бенчмарка (`results/benchmark_v2.json`, этих задач нет в обучающих данных) и на отложенные алгоритмы. Вместе с ответами исходной модели они сохраняются в `answers_<RUN_NAME>.json`.

Скомпилировать ответы можно только на Windows с Visual Prolog 11: скачайте файл ответов и запустите
`python src/eval/check_answers.py answers_<RUN_NAME>.json` — скрипт соберёт каждую программу и для алгоритмов сравнит вывод с эталоном.

In [ ]:
answers += evaluate("tuned")
tuned = [a for a in answers if a["model"] == "tuned"]
print(tuned[0]["instruction"], "\n\n", tuned[0]["answer"])

In [ ]:
#@title Задать свой вопрос { display-mode: "form" }
QUESTION = "Напиши на Visual Prolog программу, которая сортирует список слов по длине."  #@param {type:"string"}
print(ask(QUESTION))

## 8. Сохранение

Адаптер (~100–300 МБ) и ответы сохраняются на Google Диск в папку `OUT_DIR/RUN_NAME`. Модель целиком и GGUF (для Ollama и llama.cpp) — по галочкам в параметрах, они занимают несколько гигабайт.

In [ ]:
save_dir = pathlib.Path(OUT_DIR) / RUN_NAME
save_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(save_dir / "lora")
tokenizer.save_pretrained(save_dir / "lora")
json.dump(answers, open(save_dir / f"answers_{RUN_NAME}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
json.dump({"model": MODEL, "epochs": EPOCHS, "lr": LEARNING_RATE, "lora_rank": LORA_RANK, "max_seq_length": MAX_SEQ_LENGTH,
           "algo_repeat": ALGO_REPEAT, "use_pfc": USE_PFC, "use_legacy": USE_LEGACY, "data_mode": DATA_MODE, "train_examples": len(train_ds),
           "best_eval_loss": min((v for _, v in ev), default=None), "log": log},
          open(save_dir / "train_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
if SAVE_MERGED:
    model.save_pretrained_merged(str(save_dir / "merged_16bit"), tokenizer, save_method="merged_16bit")
if SAVE_GGUF:
    model.save_pretrained_gguf(str(save_dir / "gguf"), tokenizer, quantization_method="q4_k_m")
print("Сохранено в", save_dir)
!ls -la "{save_dir}"

## Что дальше

1. Скачайте `answers_<RUN_NAME>.json` с Google Диска и проверьте его компилятором: `python src/eval/check_answers.py answers_<RUN_NAME>.json`.
2. Скрипт печатает таблицу «исходная модель / обученная × бенчмарк / отложенные алгоритмы»: сколько ответов компилируется и сколько дают верный ответ.
3. Для гипотезы исследования обучите модель ещё дважды — с `DATA_MODE = "textbook"` и `"github"` (другое `RUN_NAME`) — и сравните таблицы.
4. Чтобы пользоваться моделью без Colab, включите `SAVE_GGUF` и загрузите файл `.gguf` в Ollama: `ollama create vp -f Modelfile`, где в `Modelfile` указано `FROM ./unsloth.Q4_K_M.gguf`.